# LC 42 — Trapping Rain Water
**Difficulty:** Hard &nbsp;|&nbsp; **Category:** Array
&nbsp;|&nbsp; **Pattern:** Two Pointers — Max Left/Right Walls

<div style="border-left: 4px solid purple; padding: 10px;
            background: #f9f0ff; margin: 10px 0;">
<strong>Core Insight:</strong> Water at any position equals
<code>min(max_left, max_right) - height[i]</code>.
Use two pointers and always advance the side with the
smaller max wall — that side is the bottleneck.
</div>

## Official Problem Statement

Given `n` non-negative integers representing an elevation map where
the width of each bar is `1`, compute how much water it can trap
after raining.

**Example 1:**
```
Input:  height = [0,1,0,2,1,0,1,3,2,1,2,1]
Output: 6
```

**Example 2:**
```
Input:  height = [4,2,0,3,0,2,0,1,4]
Output: 10
```

**Constraints:**
- `n == height.length`
- `1 <= n <= 2 * 10^4`
- `0 <= height[i] <= 10^5`

## What This Is Actually Asking

Imagine a row of walls of different heights.
It rains and water gets trapped between the walls.
Water can only stay if there are taller walls on both sides.
The water level at any spot is limited by the shorter of the
two surrounding walls.
Count the total units of water sitting in all the gaps.

## Walk Through an Example by Hand

Input: `height = [0,1,0,2,1,0,1,3,2,1,2,1]`

Focus on index 2 (height = 0):
- Max wall to the left of index 2  = 1 (at index 1)
- Max wall to the right of index 2 = 3 (at index 7)
- Water = min(1, 3) - 0 = 1 unit

Focus on index 4 (height = 1):
- Max wall to the left  = 2 (at index 3)
- Max wall to the right = 3 (at index 7)
- Water = min(2, 3) - 1 = 1 unit

Focus on index 5 (height = 0):
- Max wall to the left  = 2
- Max wall to the right = 3
- Water = min(2, 3) - 0 = 2 units

Two-pointer approach:
- left=0, right=11, max_l=0, max_r=0, water=0
- height[left]=0 <= height[right]=1 → process left
  max_l = max(0,0)=0, water += 0-0=0, left=1
- height[left]=1 <= height[right]=1 → process left
  max_l = max(0,1)=1, water += 1-1=0, left=2
- height[left]=0 < height[right]=1 → process left
  max_l = max(1,0)=1, water += 1-0=1, left=3
- ... continue until left >= right
- Total = 6

## The Picture

```
height = [0,1,0,2,1,0,1,3,2,1,2,1]

     3               X
     2         X     X  X     X
     1    X    X  X  X  X  X  X  X
     0 _  _  _  _  _  _  _  _  _  _  _  _
       0  1  2  3  4  5  6  7  8  9 10 11

Water fills the gaps (shown as ~):

     3               X
     2         X  ~  X  X     X
     1    X  ~ X  X  X  X  X  X  X
       0  1  2  3  4  5  6  7  8  9 10 11

Two pointers squeeze inward from both ends.
The pointer on the shorter-wall side moves first
because its wall is the bottleneck.

  L --->                        <--- R
  |  max_l tracks tallest left  |
  |  max_r tracks tallest right |
```

## When To Use This Pattern

- When the problem involves a sorted or elevation-style
  array and asks for area or volume between elements.
- When you need O(1) space — two pointers instead of
  storing a full prefix/suffix array.
- When you see "trap", "container", or "water" problems.
- When the answer at any index depends on the best value
  seen to its left AND to its right.

## The Approach

Place one pointer at the far left and one at the far right.
Track the tallest wall seen so far from each side.
At every step, move the pointer on the shorter-wall side
inward — that side limits how much water can sit there.
Add the difference between the current max wall and the
current height to the total water count.

In [1]:
from typing import List  # for type hints in the solution

In [7]:
def test_harness(func):
    """Run all test cases against func and report results."""
    tests = [
        # (input, expected, label)
        (
            [0,1,0,2,1,0,1,3,2,1,2,1],
            6,
            "LC example 1"
        ),
        (
            [4,2,0,3,0,2,0,1,4],
            20,
            "LC example 2"
        ),
        (
            [0],
            0,
            "single element — no water possible"
        ),
        (
            [3,0,3],
            3,
            "simple valley"
        ),
        (
            [1,2,3,4,5],
            0,
            "ascending — no trapping"
        ),
        (
            [5,4,3,2,1],
            0,
            "descending — no trapping"
        ),
        (
            [2,0,2],
            2,
            "symmetric valley"
        ),
        (
            [0,0,0],
            0,
            "all zeros — no walls"
        ),
    ]

    passed = 0
    for height, expected, label in tests:
        result = func(height)
        status = "PASSED" if result == expected else "FAILED"
        if status == "PASSED":
            passed += 1
        print(
            f"[{status}] {label}\n"
            f"         input={height}\n"
            f"         expected={expected}, got={result}"
        )
    print(f"\n{passed}/{len(tests)} tests passed")

In [8]:
def trap(hts: List[int]) -> int:
    """
    Compute total water trapped after rain over an elevation map.

    Approach:
    Use two pointers starting at both ends of the array.
    Track the max wall height seen from each side.
    Always advance the pointer on the shorter-wall side
    because that side determines how much water can sit.
    At each position add max_wall - height to total water.

    Time:  O(n) — each element visited once.
    Space: O(1) — only four integer variables used.
    """
    # main concept . water requires bondaries both sides to be trapped
    # First and Last Index can never trap water
    # if the left max <height> is shorter than the right; 
    # you can trap water as much as the left side max - current height
    if not hts:return 0            # edge case
    l, r = 0,len(hts) -1
    res = 0
    lMax, rMax = hts[l], hts[r]
    while l < r:
        if lMax < rMax:             # since it is water address lower first always
            l += 1
            lMax = max(lMax, hts[l])
            res += lMax - hts[l]
        else:
            r -= 1
            rMax = max(rMax, hts[r])
            res += rMax - hts[r]
    return res
    


# Quick debug prints — run this cell while building
print(trap([0,1,0,2,1,0,1,3,2,1,2,1]))  # expected: 6
print(trap([4,2,0,3,0,2,0,1,4]))          # expected: 20
print(trap([3,0,3]))                       # expected: 3
print(trap([0]))                           # expected: 0
print(trap([1,2,3,4,5]))                   # expected: 0
test_harness (trap)

6
20
3
0
0
[PASSED] LC example 1
         input=[0, 1, 0, 2, 1, 0, 1, 3, 2, 1, 2, 1]
         expected=6, got=6
[PASSED] LC example 2
         input=[4, 2, 0, 3, 0, 2, 0, 1, 4]
         expected=20, got=20
[PASSED] single element — no water possible
         input=[0]
         expected=0, got=0
[PASSED] simple valley
         input=[3, 0, 3]
         expected=3, got=3
[PASSED] ascending — no trapping
         input=[1, 2, 3, 4, 5]
         expected=0, got=0
[PASSED] descending — no trapping
         input=[5, 4, 3, 2, 1]
         expected=0, got=0
[PASSED] symmetric valley
         input=[2, 0, 2]
         expected=2, got=2
[PASSED] all zeros — no walls
         input=[0, 0, 0]
         expected=0, got=0

8/8 tests passed


In [ ]:
# Uncomment and run when solution is ready
# test_harness(trap)

## Complexity

| Approach | Time | Space |
|---|---|---|
| Brute force (nested loop) | O(n²) | O(1) |
| Prefix/suffix arrays | O(n) | O(n) |
| Two pointers (optimal) | O(n) | O(1) |

The two-pointer approach gets O(n) time AND O(1) space by
computing left/right maximums on the fly instead of
pre-storing them in arrays.

## Real World Connection

At Citi, telemetry flows in from over 6,000 endpoints.
Metric gaps — periods of silence or zero values between
spikes — behave exactly like trapped water: bounded by
the last known high reading on either side.
A Lambda function processing a Kinesis stream can use
this same two-pointer scan to estimate missing data
volume between valid DynamoDB checkpoints.
CloudWatch anomaly detection benefits from knowing the
"trapped gap" total rather than just flagging nulls.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra